# Day 4 — Pandas Fundamentals II: Cleaning, Transforming & Reshaping Data




### 1. What is the difference between .drop() and .dropna()?
.drop(): Used to remove explicitly targeted rows or columns by passing their specific index labels or names (e.g., df.drop(columns=['Rank'])).
.dropna(): Specifically scans a DataFrame to automatically drop rows or columns that contain *any* missing (NaN) values based on conditions you provide (e.g., df.dropna(subset=['Year'])).

### 2. How do you change the dtype of a column and when would you need to?
You change it using the .astype() method (e.g., df['Year'] = df['Year'].astype(int)). You need to do this when columns load with incorrect types—such as a numerical ID, count, or year importing as a continuous float (2006.0), or a numeric column importing as an object/string (which blocks arithmetic calculations).

### 3. What does .apply() do and how is it different from vectorized operations?
- Vectorized operations: Calculations happen simultaneously across entire arrays at an optimized, low-level C layer. They are extremely fast and should always be your first choice (e.g., df['A'] + df['B']).
- .apply(): Passes data elements or rows one-by-one through a custom Python function or lambda expression. It functions like a hidden loop, making it flexible for complex condition structures but significantly slower on larger datasets.

### 4. Difference between .pivot() and .pivot_table()?
- .pivot(): Reshapes data based on column values. If multiple rows share the exact same combination of row/column index pairs, it cannot reconcile them and raises a ValueError.
- .pivot_table(): An advanced generalization that can easily handle duplicate coordinate pairs because it automatically aggregates conflicting rows using a statistical summary function (default is mean, but can be sum, count, etc.).

### 5. What does .merge() do and what are the 4 types of joins?
.merge() links two DataFrames together based on shared relational keys (similar to an SQL JOIN). The 4 main types of joins are:
- Inner: Retains only rows where keys match in both DataFrames.
- Left: Retains all rows from the left DataFrame, pulling matches from the right or filling unmatched keys with NaN
- Right: Retains all rows from the right DataFrame, pulling matches from the left or filling unmatched keys with NaN.
- Outer: Retains all rows from both DataFrames, aligning keys where matches exist and filling everything else with NaN


Using the `vgsales.csv` dataset, we will execute a complete data cleaning and transformation pipeline step-by-step!

In [2]:
import pandas as pd
import numpy as np


df = pd.read_csv("vgsales.csv")
print(f"Initial Dataset Shape: {df.shape}")
df.info()

Initial Dataset Shape: (16598, 11)
<class 'pandas.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16598 non-null  int64  
 1   Name          16598 non-null  str    
 2   Platform      16598 non-null  str    
 3   Year          16327 non-null  float64
 4   Genre         16598 non-null  str    
 5   Publisher     16540 non-null  str    
 6   NA_Sales      16598 non-null  float64
 7   EU_Sales      16598 non-null  float64
 8   JP_Sales      16598 non-null  float64
 9   Other_Sales   16598 non-null  float64
 10  Global_Sales  16598 non-null  float64
dtypes: float64(6), int64(1), str(4)
memory usage: 1.4 MB


### Step 1: Identifying and Handle Missing Values
Reasoning: We will fill missing values in the `Publisher` column with `'Unknown'` to keep the sales entries. However, we will drop rows with missing `Year` values entirely, because imputing artificial release dates can severely skew time-series evaluations.

In [3]:
print("Missing values per column before:")
print(df.isna().sum())

#handling the missing data
df['Publisher'] = df['Publisher'].fillna('Unknown')
df = df.dropna(subset=['Year'])

print(f"\nShape after managing missing values: {df.shape}")

Missing values per column before:
Rank              0
Name              0
Platform          0
Year            271
Genre             0
Publisher        58
NA_Sales          0
EU_Sales          0
JP_Sales          0
Other_Sales       0
Global_Sales      0
dtype: int64

Shape after managing missing values: (16327, 11)


In [4]:
print("Missing values after cleaning: ")
print(df.isna().sum())

Missing values after cleaning: 
Rank            0
Name            0
Platform        0
Year            0
Genre           0
Publisher       0
NA_Sales        0
EU_Sales        0
JP_Sales        0
Other_Sales     0
Global_Sales    0
dtype: int64


Naisu

### Step 2: Fix Wrong Data Types
**Reasoning:** The `Year` column imports as a floating-point number (`float64`), resulting in formatting like `2006.0`. Since years are discrete calendar intervals, we convert them to integers.

In [5]:
df['Year'] = df['Year'].astype(int)
print(df[['Name', 'Platform', 'Year']].head())

                       Name Platform  Year
0                Wii Sports      Wii  2006
1         Super Mario Bros.      NES  1985
2            Mario Kart Wii      Wii  2008
3         Wii Sports Resort      Wii  2009
4  Pokemon Red/Pokemon Blue       GB  1996


### Step 3: Remove Duplicates
**Reasoning:** Duplicates can double-count units sold and misrepresent sales performance.

In [6]:
duplicate_count = df.duplicated().sum()
print(f"Identified duplicate rows: {duplicate_count}")

df = df.drop_duplicates()
print(f"Shape after dropping duplicates: {df.shape}")

Identified duplicate rows: 0
Shape after dropping duplicates: (16327, 11)


We got zero dupes

### Step 4: Create 2 New Derived Columns
**Reasoning:**
1. `International_Sales` (**Vectorized Operation**): Measures performance outside North America by computing `Global_Sales - NA_Sales`.
2. `Market_Tier` (**`.apply()` Operation**): Applies complex logic to categorize game scale based on global units sold into *Legendary*, *Blockbuster*, or *Standard* groupings.

In [7]:
# vectorized calc
df['International_Sales'] = df['Global_Sales'] - df['NA_Sales']

# fnctional assignment with .apply()
def determine_tier(sales):
    if sales > 10:
        return 'Legendary'
    elif sales >= 1:
        return 'Blockbuster'
    else:
        return 'Standard'

df['Market_Tier'] = df['Global_Sales'].apply(determine_tier)
print(df[['Name', 'Global_Sales', 'International_Sales', 'Market_Tier']].head())

                       Name  Global_Sales  International_Sales Market_Tier
0                Wii Sports         82.74                41.25   Legendary
1         Super Mario Bros.         40.24                11.16   Legendary
2            Mario Kart Wii         35.82                19.97   Legendary
3         Wii Sports Resort         33.00                17.25   Legendary
4  Pokemon Red/Pokemon Blue         31.37                20.10   Legendary


### Step 5: Reshape the Data
**Reasoning:** Aggregating metrics by `Platform` and `Genre` provides an optimized operational summary table.

In [9]:
summary_table = df.groupby(['Platform', 'Genre']).agg(
    Total_Global_Sales=('Global_Sales', 'sum'),
    Average_Global_Sales=('Global_Sales', 'mean'),
    Total_Titles_Released=('Name', 'count')
).reset_index()
print(summary_table.head(10))

  Platform       Genre  Total_Global_Sales  Average_Global_Sales  \
0     2600      Action               26.39              0.479818   
1     2600   Adventure                0.40              0.400000   
2     2600    Fighting                0.77              0.770000   
3     2600        Misc                3.58              0.716000   
4     2600    Platform               13.27              1.474444   
5     2600      Puzzle               13.65              1.706250   
6     2600      Racing                2.32              0.580000   
7     2600     Shooter               22.97              1.044091   
8     2600  Simulation                0.45              0.450000   
9     2600      Sports                2.77              0.277000   

   Total_Titles_Released  
0                     55  
1                      1  
2                      1  
3                      5  
4                      9  
5                      8  
6                      4  
7                     22  
8       

### Step 6: Merge with a Second Small Manually Created DataFrame
**Reasoning:** Merging console entries with a mapping dictionary ties hardware systems directly to their corporate platforms, providing helpful ecosystem insights for analysis.

In [ ]:
#creating secondary df mapping platforms to owners
company_mapping = {
    'Platform': ['Wii', 'NES', 'GB', 'DS', 'X360', 'PS3', 'PS2', 'GBA', 'PS4', '3DS'],
    'Parent_Company': ['Nintendo', 'Nintendo', 'Nintendo', 'Nintendo', 'Microsoft', 'Sony', 'Sony', 'Nintendo', 'Sony', 'Nintendo']
}
df_companies = pd.DataFrame(company_mapping)

#merging using a Left Join
final_dataset = pd.merge(df, df_companies, on='Platform', how='left')
final_dataset['Parent_Company'] = final_dataset['Parent_Company'].fillna('Other / PC')

print(final_dataset[['Name', 'Platform', 'Parent_Company', 'Global_Sales']].head(10))

                        Name Platform Parent_Company  Global_Sales
0                 Wii Sports      Wii       Nintendo         82.74
1          Super Mario Bros.      NES       Nintendo         40.24
2             Mario Kart Wii      Wii       Nintendo         35.82
3          Wii Sports Resort      Wii       Nintendo         33.00
4   Pokemon Red/Pokemon Blue       GB       Nintendo         31.37
5                     Tetris       GB       Nintendo         30.26
6      New Super Mario Bros.       DS       Nintendo         30.01
7                   Wii Play      Wii       Nintendo         29.02
8  New Super Mario Bros. Wii      Wii       Nintendo         28.62
9                  Duck Hunt      NES       Nintendo         28.31
